# Consolidated pipeline: RGB crop segmentation + Sequoia NDVI / NDRE

Part 1 is `01_rgb_segmentation_gli` and part 2 is `02_sequoia_alignment_indices`, unchanged and run back to back.

**What this notebook does not do:** it does not compute NDVI or NDRE per segmented crop patch. That needs the
RGB masks and the Sequoia bands on one shared pixel grid, which the current data cannot provide:

* the RGB frames come from a different camera (IMX219 parameters in the old notebook) than the Sequoia, with no
  shared georeference, and
* the 12 Sequoia frames (`IMG_260707_*`, 7 July 2026) show tree canopy over a road, not the five crops.

To get per-crop indices, either (a) fly the Sequoia over the crop plots and segment its own RGB or registered
bands, or (b) build orthomosaics of both sensors on a shared CRS and rasterise the masks onto the index grid.

# Part 1: RGB crop segmentation

# RGB crop segmentation (YOLO26n-seg) with per-patch GLI and area

Corrected version of `v4_segmentation+ indices.ipynb`.

**What was wrong in the previous version** (details in `docs/REVIEW.md`):

1. **Leaked credential.** The Roboflow API key was hard-coded in two cells of a notebook shared on
   Drive. Rotate it in Roboflow now. This notebook reads it from Colab Secrets (`ROBOFLOW_API_KEY`).
2. **Wrong weights could run silently.** The notes say the correct model is `v4_best_epoch78.pt`, but
   `segment_best.pt` was used, and `prepare()` would silently swap in any `*54*.pt` it found. The
   model is now loaded from one explicit path, and its SHA-256 and class names are logged.
3. **Class names were hard-coded** (`['fenugreek','gram','linseed','potato','wheat']`) instead of being
   read from the model. A mismatch would mislabel every patch. They are now read from
   `model.names` and checked against `data.yaml`.
4. **Masks were resized naively.** `result.masks.data` lives in the letterboxed inference frame.
   Stretching it to the image size misaligns masks for any non-square image. The notebook now uses
   `retina_masks=True` (masks at original resolution) and checks the shape.
5. **Overlapping instances were double-counted** in area and yield. Each pixel now belongs to one
   instance only (the highest-confidence one).
6. **The GSD block was inconsistent.** Footprint / 640 was applied to masks resized to the image's
   own size. GSD is now computed from each image's actual width and height. This is valid only if the
   dataset images are full camera frames (resized, not cropped or tiled). See `IMAGES_ARE_FULL_FRAMES`.
7. **Mask GLI mixed soil and canopy.** The mean GLI over a crop polygon depends on canopy cover. The
   notebook now reports the vegetation fraction and GLI over vegetation pixels separately.
8. NaN GLI was silently mapped to `MODERATE`; it is now `UNKNOWN`. `cv2.putText` cannot draw `²`,
   so labels use `m2`.
9. **No accuracy was ever computed.** The notebook only drew predictions. It now runs `model.val()` on the test
   split (mask mAP) and checks for near-duplicate images across train/valid/test (leakage).
10. The "NDVI" part ran on a third-party paddy dataset (IIT Tirupati `Set_1/Set_2`) with the same
   broken radiometry. It is removed here. Sequoia indices live in notebook 02.

**Read before quoting numbers.** The GLI condition thresholds and the yield factors are not
validated against any field measurement, so "yield" here is a scenario (reference yield x area x
factor), not an estimate. It is labelled `indicative` in every output.

In [ ]:
# ---------------------------------------------------------------- setup
import os, sys, csv, json, hashlib, subprocess, shutil, warnings
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'roboflow'], check=True)

import numpy as np
import cv2
import yaml
import torch
from ultralytics import YOLO

In [ ]:
# ---------------------------------------------------------------- configuration
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()

# Weights: upload ONE file and point to it. The notes say v4_best_epoch78.pt is the correct model.
MODEL_PATH = Path(os.environ.get('SEG_MODEL', str(ROOT / 'v4_best_epoch78.pt')))

# Roboflow export (YOLO segmentation format)
RF_WORKSPACE, RF_PROJECT, RF_VERSION, RF_FORMAT = 'kavadis-workspace', 'crop_classification_orin', 4, 'yolo26'
DATASET_DIR = Path(os.environ.get('SEG_DATASET', str(ROOT / f'{RF_PROJECT}-{RF_VERSION}')))
EVAL_SPLIT = 'test'

RESULTS = Path(os.environ.get('SEG_OUT', str(ROOT / 'Results' / 'Segmentation')))
IMGSZ, CONF = 640, 0.25

# Ground scale. These are the Raspberry Pi Camera v2 (IMX219) values from the old notebook.
# Replace them with the camera and altitude that actually took the images.
CAMERA = {'sensor_w_mm': 3.68, 'sensor_h_mm': 2.76, 'focal_mm': 3.04, 'height_agl_m': 20.0}
IMAGES_ARE_FULL_FRAMES = True   # False if Roboflow cropped or tiled the frames: area is then meaningless

# GLI vegetation mask and condition classes (illustrative, NOT validated)
GLI_VEG_THRESHOLD = 0.0
GLI_THRESHOLDS = {'wheat': (0.05, 0.10, 0.15), 'potato': (0.04, 0.08, 0.13), 'gram': (0.03, 0.07, 0.12),
                  'fenugreek': (0.05, 0.09, 0.14), 'linseed': (0.05, 0.09, 0.14)}
REFERENCE_YIELD_KG_HA = {'wheat': 3547, 'potato': 24629, 'gram': 1199, 'fenugreek': 1582, 'linseed': 633}
YIELD_FACTORS = {'STRESSED / POOR': 0.50, 'MODERATE': 0.75, 'GOOD': 1.00, 'VERY GOOD': 1.10, 'UNKNOWN': np.nan}

PALETTE = [(0, 255, 255), (0, 255, 0), (255, 0, 255), (255, 255, 0), (0, 128, 255), (255, 0, 0), (0, 0, 255)]
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

In [ ]:
# ---------------------------------------------------------------- data and model
def download_dataset():
    """Download the Roboflow export once. The key comes from Colab Secrets or the environment, never the notebook."""
    if (DATASET_DIR / 'data.yaml').is_file():
        return DATASET_DIR
    key = os.environ.get('ROBOFLOW_API_KEY')
    if not key and IN_COLAB:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
    if not key:
        raise RuntimeError('Set ROBOFLOW_API_KEY in Colab Secrets (key icon, left sidebar).')
    from roboflow import Roboflow
    ds = Roboflow(api_key=key).workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        RF_FORMAT, location=str(DATASET_DIR))
    return Path(ds.location)


def dataset_yaml_abs(root):
    """Roboflow's data.yaml uses '../train/images' paths; write a copy with paths Ultralytics resolves reliably."""
    cfg = yaml.safe_load(open(root / 'data.yaml'))
    names = cfg['names']
    names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
    split_dir = lambda s: next((f'{a}/images' for a in ([s] if s != 'valid' else ['valid', 'val'])
                                if (root / a / 'images').is_dir()), None)
    out = {'path': str(root.resolve()), 'names': {i: n for i, n in enumerate(names)}}
    for k, s in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
        if split_dir(s):
            out[k] = split_dir(s)
    p = root / 'data_abs.yaml'
    yaml.safe_dump(out, open(p, 'w'), sort_keys=False)
    return p, names


def sha256(path, n=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(n), b''):
            h.update(chunk)
    return h.hexdigest()


def load_model(path, expected_names=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'{path} not found. Upload the weights and set MODEL_PATH; no fallback is attempted.')
    model = YOLO(str(path))
    if model.task != 'segment':
        raise ValueError(f'{path.name}: task {model.task!r}, expected segment')
    names = [model.names[i] for i in sorted(model.names)]
    print(f'model  : {path.name}  sha256={sha256(path)[:16]}  classes={names}')
    if expected_names is not None and list(expected_names) != names:
        raise ValueError(f'class order mismatch: model {names} vs data.yaml {list(expected_names)}')
    return model, names

In [ ]:
# ---------------------------------------------------------------- split leakage check
def dhash(path, size=8):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).ravel())


def leakage_report(root, max_hamming=6):
    """Near-duplicate images across splits (consecutive UAV frames, Roboflow augment copies).
    Any hit means the test score is optimistic."""
    splits = {s: sorted(p for p in (root / s / 'images').iterdir() if p.suffix.lower() in IMG_EXT)
              for s in ('train', 'valid', 'test') if (root / s / 'images').is_dir()}
    hashes = {s: np.stack([dhash(p) for p in ps]) for s, ps in splits.items() if ps}
    rows = []
    for a, b in (('train', 'test'), ('train', 'valid'), ('valid', 'test')):
        if a in hashes and b in hashes:
            d = np.unpackbits(hashes[b][:, None, :] ^ hashes[a][None, :, :], axis=2).sum(2)
            nearest = d.min(1)
            rows.append({'pair': f'{b} vs {a}', f'{b}_images': len(nearest),
                         f'near_duplicates(<={max_hamming})': int((nearest <= max_hamming).sum()),
                         'median_nearest_hamming': float(np.median(nearest))})
    for r in rows:
        print(r)
    return rows

In [ ]:
# ---------------------------------------------------------------- ground scale
def area_per_pixel_m2(img_w, img_h):
    if not IMAGES_ARE_FULL_FRAMES:
        return float('nan')
    fw = CAMERA['height_agl_m'] * CAMERA['sensor_w_mm'] / CAMERA['focal_mm']   # footprint width, m
    fh = CAMERA['height_agl_m'] * CAMERA['sensor_h_mm'] / CAMERA['focal_mm']
    return (fw / img_w) * (fh / img_h)

In [ ]:
# ---------------------------------------------------------------- per-image analysis
def gli_map(bgr):
    b, g, r = [bgr[..., i].astype(np.float32) for i in range(3)]
    den = 2 * g + r + b
    with np.errstate(divide='ignore', invalid='ignore'):
        out = (2 * g - r - b) / den
    out[den <= 0] = np.nan
    return out


def condition(cls, gli):
    if not np.isfinite(gli) or cls not in GLI_THRESHOLDS:
        return 'UNKNOWN'
    t1, t2, t3 = GLI_THRESHOLDS[cls]
    return 'STRESSED / POOR' if gli < t1 else 'MODERATE' if gli < t2 else 'GOOD' if gli < t3 else 'VERY GOOD'


def instance_ownership(result, shape):
    """-> (owner map: -1 or instance index, per-instance boolean masks). Highest confidence wins overlaps."""
    h, w = shape
    owner = np.full((h, w), -1, np.int32)
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return owner, []
    m = result.masks.data.cpu().numpy() > 0.5
    if m.shape[1:] != (h, w):           # should not happen with retina_masks=True
        m = np.stack([cv2.resize(x.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0 for x in m])
        warnings.warn('mask resolution differed from image; resized (check retina_masks)')
    conf = result.boxes.conf.cpu().numpy()
    for i in np.argsort(conf):           # low confidence first, overwritten by higher
        owner[m[i]] = i
    return owner, m


def draw_label(img, text, x, y, placed):
    font, fs, th = cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
    (tw, tht), base = cv2.getTextSize(text, font, fs, th)
    H, W = img.shape[:2]
    for dx, dy in ((-tw // 2, -10), (-tw // 2, tht + 20), (12, 0), (-tw - 12, 0)):
        x0 = int(np.clip(x + dx, 4, W - tw - 4)); y0 = int(np.clip(y + dy, tht + 4, H - base - 4))
        box = (x0 - 4, y0 - tht - 4, x0 + tw + 4, y0 + base + 4)
        if not any(not (box[2] < b[0] or b[2] < box[0] or box[3] < b[1] or b[3] < box[1]) for b in placed):
            break
    cv2.rectangle(img, box[:2], box[2:], (0, 0, 0), -1)
    cv2.putText(img, text, (x0, y0), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    placed.append(box)


def analyse_image(path, result, names):
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    owner, _ = instance_ownership(result, (h, w))
    gli = gli_map(img)
    a_px = area_per_pixel_m2(w, h)
    vis, placed, rows = img.copy(), [], []
    if result.boxes is None or len(result.boxes) == 0:
        return vis, rows
    cls_ids = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    for i in range(len(cls_ids)):
        own = owner == i
        npx = int(own.sum())
        if npx == 0:
            continue                      # fully covered by a higher-confidence instance
        name = names[cls_ids[i]]
        g = gli[own]; g = g[np.isfinite(g)]
        veg = g > GLI_VEG_THRESHOLD
        gli_all = float(g.mean()) if g.size else np.nan
        gli_veg = float(g[veg].mean()) if veg.any() else np.nan
        cond = condition(name, gli_all)
        area = npx * a_px
        rate = REFERENCE_YIELD_KG_HA.get(name, np.nan) / 1e4 * YIELD_FACTORS[cond]
        rows.append({'image': path.name, 'patch': i + 1, 'class': name, 'confidence': float(confs[i]),
                     'pixels': npx, 'area_m2': area, 'veg_fraction': float(veg.mean()) if g.size else np.nan,
                     'gli_mean': gli_all, 'gli_median': float(np.median(g)) if g.size else np.nan,
                     'gli_std': float(g.std()) if g.size else np.nan, 'gli_mean_veg_only': gli_veg,
                     'condition_indicative': cond, 'indicative_yield_kg': area * rate})
        col = np.array(PALETTE[cls_ids[i] % len(PALETTE)], np.float32)
        vis[own] = (vis[own].astype(np.float32) * 0.55 + col * 0.45).astype(np.uint8)
        ys, xs = np.nonzero(own)
        draw_label(vis, f'#{i + 1} {name} {confs[i]:.2f} A:{area:.1f}m2 GLI:{gli_all:.2f}',
                   int(xs.mean()), int(ys.mean()), placed)
    return vis, rows

In [ ]:
# ---------------------------------------------------------------- run
def run_segmentation(model, names, image_dir, out_dir=RESULTS):
    out_dir = Path(out_dir); (out_dir / 'overlays').mkdir(parents=True, exist_ok=True)
    images = sorted(p for p in Path(image_dir).iterdir() if p.suffix.lower() in IMG_EXT)
    if not images:
        raise FileNotFoundError(f'no images in {image_dir}')
    dev = 0 if torch.cuda.is_available() else 'cpu'
    rows = []
    for path in images:
        r = model.predict(str(path), imgsz=IMGSZ, conf=CONF, device=dev, retina_masks=True, verbose=False)[0]
        vis, rr = analyse_image(path, r, names)
        cv2.imwrite(str(out_dir / 'overlays' / path.name), vis)
        rows += rr
    keys = list(rows[0].keys()) if rows else ['image']
    with open(out_dir / 'segmentation_patches.csv', 'w', newline='') as f:
        wr = csv.DictWriter(f, keys); wr.writeheader(); wr.writerows(rows)

    total_area = sum(r['area_m2'] for r in rows)
    by_class = {}
    for r in rows:
        d = by_class.setdefault(r['class'], {'patches': 0, 'area_m2': 0.0, 'gli': []})
        d['patches'] += 1; d['area_m2'] += r['area_m2']; d['gli'].append(r['gli_mean'])
    with open(out_dir / 'SEGMENTATION_REPORT.txt', 'w') as f:
        f.write(f'YOLO-seg crop segmentation report\n{"=" * 70}\n')
        f.write(f'model {MODEL_PATH.name} | images {len(images)} | patches {len(rows)} | conf {CONF} | imgsz {IMGSZ}\n')
        f.write(f'camera {CAMERA} | full frames: {IMAGES_ARE_FULL_FRAMES}\n')
        f.write('Yield and condition are INDICATIVE (unvalidated thresholds/factors).\n\n')
        for c, d in sorted(by_class.items()):
            f.write(f'{c:<10s} patches {d["patches"]:3d}  area {d["area_m2"]:10.2f} m2  '
                    f'mean GLI {np.nanmean(d["gli"]):.3f}\n')
        f.write(f'\nTotal area {total_area:.2f} m2\n')
    print(f'{len(images)} images, {len(rows)} patches, total area {total_area:.1f} m2 -> {out_dir}')
    return rows


def evaluate(model, data_yaml, split=EVAL_SPLIT):
    """Mask/box mAP on a held-out split. Report this, not the training-time valid score."""
    m = model.val(data=str(data_yaml), split=split, imgsz=IMGSZ, batch=8, plots=False, verbose=False,
                  device=0 if torch.cuda.is_available() else 'cpu')
    res = {'mask_mAP50': float(m.seg.map50), 'mask_mAP50_95': float(m.seg.map),
           'box_mAP50_95': float(m.box.map),
           'per_class_mask_mAP50_95': {model.names[int(c)]: float(v) for c, v in zip(m.seg.ap_class_index, m.seg.maps[m.seg.ap_class_index])}}
    print(json.dumps(res, indent=2))
    return res

In [ ]:
if os.environ.get('SEG_SKIP_RUN') != '1':
    ds_root = download_dataset()
    data_yaml, ds_names = dataset_yaml_abs(ds_root)
    leakage_report(ds_root)
    seg_model, seg_names = load_model(MODEL_PATH, ds_names)
    seg_metrics = evaluate(seg_model, data_yaml)
    split_dir = ds_root / EVAL_SPLIT / 'images'
    seg_rows = run_segmentation(seg_model, seg_names, split_dir)

# Part 2: Sequoia calibration, alignment, NDVI and NDRE

# Parrot Sequoia: radiometric calibration, band alignment, NDVI and NDRE

Corrected version of `12_parrot_images (corrected).ipynb`.

**What was wrong in the previous version** (details in `docs/REVIEW.md`):

1. **Radiometry was a no-op.** The code asked exiftool for the tags `Black Level`, `Sensor Gain` and
   `Irradiance`. Sequoia files have none of them, so every band fell back to black = 64, gain = 1,
   irradiance = 1 (visible in the old output). The real dark level is about 4,300 to 4,600 DN (the
   `SensorModel` B coefficient) and band sensitivity differs about 3x between RED and REG
   (coefficient A). The old NDVI and NDRE were therefore ratios of raw DN. That is why mean NDRE
   came out *higher* than mean NDVI, which is physically backwards for vegetation.
2. **No vignetting correction.** Corners are 17 to 30 % darker than the centre (`VignettingPolynomial2D`).
3. Alignment bugs: affine composition in the wrong order (NIR -> REG -> RED), NCC scored on the
   zero-filled border, and failed captures fell back to *identity* (about 20 px misregistration)
   instead of the rig's known transform.
4. NDVI category thresholds (0.2 / 0.5) were reused for NDRE, which has a different range.
5. "Reflectance" CSVs were each band divided by its own maximum, which destroys the band ratios.
   They are removed.

**What this notebook does**

1. Reads the calibration metadata straight from each TIFF (XMP + EXIF), with no exiftool.
2. DN -> at-sensor irradiance with Parrot's model `I = f^2 (p - B) / (A*t*ISO + C)`, then divides by the
   vignetting polynomial.
3. Optional illumination normalisation from the Sunshine sensor, and optional reflectance-panel
   calibration.
4. Registers the bands (gradient images, phase-correlation start, coarse-to-fine ECC homography), checks
   every capture against the flight median, and falls back to the median transform when a capture fails.
5. Computes NDVI and NDRE on the common valid area and saves GeoTIFFs, PNGs, QC overlays and CSVs.

**Read before quoting numbers:** without a reflectance panel the indices are computed from
calibrated at-sensor irradiance, not surface reflectance. They are consistent within a flight but are
not comparable to literature thresholds. Photograph the panel before and after each flight and fill in
`PANEL` below.

In [ ]:
# ---------------------------------------------------------------- setup
import os, sys, re, json, base64, struct, subprocess, warnings
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio', 'pillow', 'opencv-python-headless'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

import numpy as np
import pandas as pd
import cv2
import rasterio
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ExifTags

warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

In [ ]:
# ---------------------------------------------------------------- configuration
SEQ_IMAGE_FOLDER = Path(os.environ.get('SEQUOIA_DIR', '/content/drive/MyDrive/parrot 12 images'))
SEQ_OUT_ROOT = Path(os.environ.get('SEQUOIA_OUT', str(SEQ_IMAGE_FOLDER / 'Calibrated_Aligned_Indices')))

SEQ_BAND_SUFFIXES = {'GRE': ['GRE'], 'RED': ['RED'], 'REG': ['REG'], 'NIR': ['NIR']}
SEQ_BAND_ORDER = ['GRE', 'RED', 'REG', 'NIR']
SEQ_REFERENCE_BAND = 'REG'      # spectrally between RED and NIR: the best-correlated partner for both

# Radiometry
NORMALISE_ILLUMINATION = True   # scale each capture by Sunshine irradiance relative to the flight median
SUNSHINE_GAIN = {0: 1.0, 1: 4.0, 2: 16.0, 3: 64.0}   # gain index -> gain; only matters if indices differ
# Reflectance panel (set to None if no panel image was taken).
# 'capture' is the capture prefix of the panel shot, 'roi' = (row0, row1, col0, col1) INSIDE the panel
# in the aligned frame, 'reflectance' = the panel's certified reflectance per band.
PANEL = None
# PANEL = {'capture': 'IMG_260707_103000_0001', 'roi': (400, 520, 560, 700),
#          'reflectance': {'GRE': 0.18, 'RED': 0.18, 'REG': 0.18, 'NIR': 0.18}}

# Registration
PYRAMID_LEVELS = 4
ECC_MAX_ITER = 300
ECC_EPS = 1e-6
GRAD_BLUR_SIGMA = 1.5
MIN_ECC_CC = 0.30               # below this a capture is flagged
MAX_DEV_FROM_MEDIAN_PX = 3.0    # corner deviation from the flight-median transform before fallback

# Index category thresholds: illustrative only, calibrate against field data
INDEX_CLASSES = {
    'NDVI': (0.2, 0.5),
    'NDRE': (0.1, 0.3),
}

In [ ]:
# ---------------------------------------------------------------- metadata (XMP + EXIF, no exiftool)
def _xmp_text(path):
    raw = Path(path).read_bytes()
    s, e = raw.find(b'<x:xmpmeta'), raw.find(b'</x:xmpmeta>')
    if s < 0 or e < 0:
        raise ValueError(f'{path}: no XMP block; is this an original Sequoia TIFF?')
    return raw[s:e].decode('utf-8', 'replace')


def _xmp_attr(xmp, key):
    m = re.search(rf'Camera:{key}="([^"]*)"', xmp)
    return m.group(1) if m else None


def _xmp_seq(xmp, key):
    m = re.search(rf'<Camera:{key}>.*?<rdf:li>(.*?)</rdf:li>', xmp, re.S)
    return m.group(1).strip() if m else None


def _floats(s):
    return [float(v) for v in s.split(',')] if s else None


def read_sequoia_meta(path):
    """All calibration fields needed downstream. Raises if a required field is missing,
    instead of silently falling back to defaults (the old code's failure mode)."""
    xmp = _xmp_text(path)
    im = Image.open(path)
    exif = im.getexif()
    tags = {ExifTags.TAGS.get(k, k): v for k, v in list(exif.items()) + list(exif.get_ifd(0x8769).items())}

    sensor_model = _floats(_xmp_seq(xmp, 'SensorModel'))
    vig_pw = _xmp_seq(xmp, 'VignettingPolynomial2DName')
    vig_c = _floats(_xmp_seq(xmp, 'VignettingPolynomial2D'))
    meta = {
        'path': str(path),
        'band_name': _xmp_seq(xmp, 'BandName'),
        'exposure_s': float(tags.get('ExposureTime', 'nan')),
        'iso': float(tags.get('ISOSpeedRatings', 'nan')),
        'fnumber': float(tags.get('FNumber', 'nan')),
        'sensor_model': sensor_model,                     # A, B, C
        'vig_powers': [int(v) for v in vig_pw.split(',')] if vig_pw else None,
        'vig_coeffs': vig_c,
        'camera_ypr_deg': [_floats(_xmp_attr(xmp, k))[0] if _xmp_attr(xmp, k) else np.nan
                           for k in ('Yaw', 'Pitch', 'Roll')],
        'irradiance': None,
    }
    required = ['sensor_model', 'vig_powers', 'vig_coeffs']
    missing = [k for k in required if not meta[k]] + \
              [k for k in ('exposure_s', 'iso', 'fnumber') if not np.isfinite(meta[k])]
    if missing:
        raise ValueError(f'{Path(path).name}: missing calibration metadata {missing}')

    b64 = re.search(r'<Camera:IrradianceList>\s*(.*?)\s*</Camera:IrradianceList>', xmp, re.S)
    if b64:
        raw = base64.b64decode(b64.group(1).strip())
        recs = [struct.unpack('<QHHHH3f', raw[i:i + 28]) for i in range(0, len(raw) - 27, 28)]
        # (timestamp_us, ch0_count, ch1_count, gain_index, integration_ms, yaw, pitch, roll)
        if recs:
            meta['irradiance'] = {
                'gain_index': sorted({r[3] for r in recs}),
                'value': float(np.median([r[1] / (SUNSHINE_GAIN[r[3]] * r[4]) for r in recs])),
                'min_count': int(min(r[1] for r in recs)),
            }
    return meta

In [ ]:
# ---------------------------------------------------------------- radiometry
def vignetting_map(meta, shape):
    """Parrot 2-D vignetting polynomial on normalised coordinates x=col/W, y=row/H."""
    h, w = shape
    x = (np.arange(w, dtype=np.float64) / w)[None, :]
    y = (np.arange(h, dtype=np.float64) / h)[:, None]
    p = meta['vig_powers']
    v = np.zeros(shape, np.float64)
    for c, i, j in zip(meta['vig_coeffs'], p[0::2], p[1::2]):
        v += c * x ** i * y ** j
    return v.astype(np.float32)


def dn_to_sensor_irradiance(dn, meta):
    """Parrot Sequoia sensor model:  I = f^2 * (p - B) / (A * t * ISO + C), then / vignetting."""
    A, B, C = meta['sensor_model']
    f, t, iso = meta['fnumber'], meta['exposure_s'], meta['iso']
    I = f * f * (dn.astype(np.float32) - B) / (A * t * iso + C)
    return I / vignetting_map(meta, dn.shape)

In [ ]:
# ---------------------------------------------------------------- registration
def _unit(a):
    a = np.nan_to_num(a, nan=float(np.nanmedian(a)))
    lo, hi = np.percentile(a, (1, 99))
    return np.clip((a - lo) / (hi - lo + 1e-12), 0, 1).astype(np.float32)


def gradient_image(a):
    g = cv2.GaussianBlur(_unit(a), (0, 0), GRAD_BLUR_SIGMA)
    m = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3))
    return (m / (m.max() + 1e-12)).astype(np.float32)


def warp(img, W, shape, interp=cv2.INTER_LINEAR, border=0.0):
    h, w = shape
    return cv2.warpPerspective(img, W, (w, h), flags=interp | cv2.WARP_INVERSE_MAP,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=border)


def ncc(a, b, mask=None):
    if mask is not None:
        a, b = a[mask], b[mask]
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a * a).sum() * (b * b).sum()) + 1e-12))


def register(ref_g, mov_g):
    """Homography W with warp(mov, W) ~ ref. Phase-correlation start, coarse-to-fine ECC."""
    pr, pm = [ref_g], [mov_g]
    for _ in range(PYRAMID_LEVELS - 1):
        pr.append(cv2.pyrDown(pr[-1])); pm.append(cv2.pyrDown(pm[-1]))
    pr, pm = pr[::-1], pm[::-1]
    (dx, dy), _ = cv2.phaseCorrelate(pr[0], pm[0])
    cands = [np.eye(3, dtype=np.float32)]
    for s in (1.0, -1.0):
        Wc = np.eye(3, dtype=np.float32); Wc[0, 2], Wc[1, 2] = s * dx, s * dy
        cands.append(Wc)
    W = cands[int(np.argmax([ncc(pr[0], warp(pm[0], c, pr[0].shape)) for c in cands]))]
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, ECC_MAX_ITER, ECC_EPS)
    cc = float('nan')
    S = np.diag([0.5, 0.5, 1.0])
    for lvl, (r, m) in enumerate(zip(pr, pm)):
        try:
            cc, W = cv2.findTransformECC(r, m, W.astype(np.float32), cv2.MOTION_HOMOGRAPHY, crit, None, 5)
        except cv2.error:
            pass
        if lvl < len(pr) - 1:
            W = np.linalg.inv(S) @ W @ S
            W = (W / W[2, 2]).astype(np.float32)
    return W.astype(np.float32), float(cc)


def corner_deviation(W1, W2, shape):
    h, w = shape
    pts = np.array([[0, 0, 1], [w - 1, 0, 1], [0, h - 1, 1], [w - 1, h - 1, 1], [w / 2, h / 2, 1]], np.float64).T
    a, b = W1 @ pts, W2 @ pts
    return float(np.max(np.linalg.norm(a[:2] / a[2] - b[:2] / b[2], axis=0)))


def centre_shift(W, shape):
    h, w = shape
    p = W @ np.array([w / 2, h / 2, 1.0])
    return float(p[0] / p[2] - w / 2), float(p[1] / p[2] - h / 2)

In [ ]:
# ---------------------------------------------------------------- indices and output helpers
def normalized_difference(a, b, valid):
    with np.errstate(divide='ignore', invalid='ignore'):
        nd = (a - b) / (a + b)
    ok = valid & (a > 0) & (b > 0) & np.isfinite(nd)
    out = np.full(a.shape, np.nan, np.float32)
    out[ok] = nd[ok]
    return out


def class_breakdown(index, thresholds):
    v = index[np.isfinite(index)]
    lo, hi = thresholds
    n = max(v.size, 1)
    return {'low_%': 100 * np.sum(v < lo) / n, 'mid_%': 100 * np.sum((v >= lo) & (v <= hi)) / n,
            'high_%': 100 * np.sum(v > hi) / n}


def save_tif(path, arrays, names, tags=None):
    arrays = [np.asarray(a, np.float32) for a in arrays]
    h, w = arrays[0].shape
    with rasterio.open(path, 'w', driver='GTiff', height=h, width=w, count=len(arrays),
                       dtype='float32', nodata=np.nan, compress='deflate') as dst:
        for i, (a, n) in enumerate(zip(arrays, names), 1):
            dst.write(a, i); dst.set_band_description(i, n)
        if tags:
            dst.update_tags(**{k: str(v) for k, v in tags.items()})


def save_index_png(index, title, path, vmin, vmax):
    fig, ax = plt.subplots(figsize=(7, 5.4))
    cmap = matplotlib.colormaps['RdYlGn'].copy(); cmap.set_bad('black')
    im = ax.imshow(index, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title); ax.axis('off')
    fig.savefig(path, bbox_inches='tight', dpi=150); plt.close(fig)


def false_colour(bands, order):
    out = []
    for b in order:
        a = np.nan_to_num(bands[b], nan=0.0)
        lo, hi = np.nanpercentile(bands[b], (1, 99))
        out.append(np.clip((a - lo) / (hi - lo + 1e-12), 0, 1))
    return np.dstack(out)


def save_qc(name, before, after, path):
    order = ['NIR', 'RED', 'GRE' if 'GRE' in after else 'REG']
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    ax[0].imshow(false_colour(before, order)); ax[0].set_title(f'{name}: before registration')
    ax[1].imshow(false_colour(after, order)); ax[1].set_title(f'{name}: registered (colour fringes = residual error)')
    for a in ax:
        a.axis('off')
    fig.savefig(path, bbox_inches='tight', dpi=120); plt.close(fig)

In [ ]:
# ---------------------------------------------------------------- pipeline
def find_capture_sets(folder):
    lut = {s.upper(): b for b, sufs in SEQ_BAND_SUFFIXES.items() for s in sufs}
    pat = re.compile(r'^(?P<prefix>.+?)_(?P<band>' + '|'.join(sorted(lut, key=len, reverse=True)) + r')$', re.I)
    sets = {}
    for p in sorted(Path(folder).iterdir()):
        if p.is_file() and p.suffix.lower() in {'.tif', '.tiff'}:
            m = pat.match(p.stem)
            if m:
                sets.setdefault(m.group('prefix'), {})[lut[m.group('band').upper()]] = p
    return sets


def calibrate(files):
    """-> metas, sensor-irradiance bands (float32), shape."""
    metas = {b: read_sequoia_meta(files[b]) for b in files}
    dn = {b: cv2.imread(str(files[b]), cv2.IMREAD_UNCHANGED) for b in files}
    shapes = {a.shape for a in dn.values()}
    if len(shapes) != 1:
        raise ValueError(f'band shapes differ: {shapes}')
    bands = {b: dn_to_sensor_irradiance(dn[b], metas[b]) for b in files}
    return metas, bands, shapes.pop()


def run_sequoia_pipeline(folder=None, out_root=None):
    folder = Path(folder or SEQ_IMAGE_FOLDER)
    out_root = Path(out_root or SEQ_OUT_ROOT)
    dirs = {k: out_root / k for k in ('Calibrated_Bands', 'NDVI', 'NDRE', 'Alignment_QC')}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

    sets = find_capture_sets(folder)
    sets = {k: v for k, v in sets.items() if SEQ_REFERENCE_BAND in v and len(v) >= 2}
    if not sets:
        raise FileNotFoundError(f'No Sequoia capture sets with a {SEQ_REFERENCE_BAND} band in {folder}')
    print(f'{len(sets)} capture set(s) in {folder}')

    # ---- pass 1: radiometry and per-capture registration
    cap = {}
    for name, files in sets.items():
        metas, bands, shape = calibrate(files)
        ref_g = gradient_image(bands[SEQ_REFERENCE_BAND])
        reg = {}
        for b in bands:
            if b == SEQ_REFERENCE_BAND:
                continue
            W, cc = register(ref_g, gradient_image(bands[b]))
            reg[b] = {'W': W, 'cc': cc}
        cap[name] = {'files': files, 'metas': metas, 'bands': bands, 'shape': shape, 'reg': reg}
        print(f'  {name}: ' + ', '.join(f'{b} cc={r["cc"]:.3f}' for b, r in reg.items()))

    # ---- pass 2: rig consistency. The bands are rigidly mounted, so transforms should agree across
    # captures at similar altitude. Captures that disagree with the flight median use the median.
    bands_all = sorted({b for c in cap.values() for b in c['reg']})
    median_W = {}
    for b in bands_all:
        Ws = [c['reg'][b]['W'] for c in cap.values() if b in c['reg'] and c['reg'][b]['cc'] >= MIN_ECC_CC]
        if Ws:
            median_W[b] = np.median(np.stack(Ws), axis=0).astype(np.float32)

    # ---- illumination reference per band (flight median of Sunshine irradiance)
    sun_ref = {}
    for b in SEQ_BAND_ORDER:
        vals = [c['metas'][b]['irradiance']['value'] for c in cap.values()
                if b in c['metas'] and c['metas'][b]['irradiance']]
        if vals:
            sun_ref[b] = float(np.median(vals))
    gains = {b: c['metas'][b]['irradiance']['gain_index'] for c in cap.values() for b in c['metas']
             if c['metas'][b]['irradiance']}
    if len({tuple(g) for g in gains.values()}) > 1:
        print('  NOTE: Sunshine gain index varies; check SUNSHINE_GAIN mapping before trusting normalisation')

    align_rows, index_rows = [], []
    for name, c in cap.items():
        shape, bands = c['shape'], c['bands']
        ref_g = gradient_image(bands[SEQ_REFERENCE_BAND])
        aligned = {SEQ_REFERENCE_BAND: bands[SEQ_REFERENCE_BAND]}
        valid = np.isfinite(bands[SEQ_REFERENCE_BAND])
        for b, r in c['reg'].items():
            W, used = r['W'], 'ECC homography'
            dev = corner_deviation(W, median_W[b], shape) if b in median_W else float('nan')
            if r['cc'] < MIN_ECC_CC or (np.isfinite(dev) and dev > MAX_DEV_FROM_MEDIAN_PX):
                if b in median_W:
                    W, used = median_W[b], 'flight-median fallback'
                print(f'  {name} {b}: cc={r["cc"]:.3f}, deviation {dev:.1f} px -> {used}')
            mov_g = gradient_image(bands[b])
            out = warp(bands[b], W, shape, border=np.nan)
            ok = np.isfinite(out)
            ncc_before = ncc(ref_g, mov_g, valid & np.isfinite(bands[b]))
            ncc_after = ncc(ref_g, warp(mov_g, W, shape), valid & ok)
            aligned[b] = out
            valid &= ok
            sx, sy = centre_shift(W, shape)
            align_rows.append({'capture': name, 'band': b, 'reference': SEQ_REFERENCE_BAND, 'method': used,
                               'ecc_cc': r['cc'], 'grad_ncc_before': ncc_before, 'grad_ncc_after': ncc_after,
                               'centre_shift_x_px': sx, 'centre_shift_y_px': sy,
                               'dev_from_flight_median_px': dev, 'homography': json.dumps(np.round(W, 6).tolist())})
        # common valid area -> NaN elsewhere (no cropping, so every capture keeps the same pixel grid)
        for b in aligned:
            aligned[b] = np.where(valid, aligned[b], np.nan).astype(np.float32)

        # illumination normalisation (Sunshine), relative to the flight median per band
        level = 'sensor_irradiance'
        if NORMALISE_ILLUMINATION and all(b in sun_ref and c['metas'][b]['irradiance'] for b in aligned):
            for b in aligned:
                aligned[b] = aligned[b] * (sun_ref[b] / c['metas'][b]['irradiance']['value'])
            level = 'sensor_irradiance_sun_normalised'
        c['aligned'], c['valid'], c['level'] = aligned, valid, level

    # ---- optional panel calibration -> reflectance
    K_panel = None
    if PANEL:
        pc = cap.get(PANEL['capture'])
        if pc is None:
            raise KeyError(f'panel capture {PANEL["capture"]} not found')
        r0, r1, c0, c1 = PANEL['roi']
        K_panel = {b: PANEL['reflectance'][b] / float(np.nanmean(pc['aligned'][b][r0:r1, c0:c1]))
                   for b in pc['aligned']}
        print('  panel factors:', {b: f'{k:.4g}' for b, k in K_panel.items()})

    for name, c in cap.items():
        aligned, valid, level = c['aligned'], c['valid'], c['level']
        if K_panel:
            aligned = {b: a * K_panel[b] for b, a in aligned.items()}
            level = 'reflectance_panel'
        names = [b for b in SEQ_BAND_ORDER if b in aligned]
        save_tif(dirs['Calibrated_Bands'] / f'{name}_bands.tif', [aligned[b] for b in names], names,
                 {'radiometric_level': level, 'reference_band': SEQ_REFERENCE_BAND})
        save_qc(name, c['bands'], aligned, dirs['Alignment_QC'] / f'{name}_alignment_QC.png')

        row = {'capture': name, 'radiometric_level': level, 'valid_area_%': 100 * valid.mean()}
        idx = {}
        if 'RED' in aligned and 'NIR' in aligned:
            idx['NDVI'] = normalized_difference(aligned['NIR'], aligned['RED'], valid)
        if 'REG' in aligned and 'NIR' in aligned:
            idx['NDRE'] = normalized_difference(aligned['NIR'], aligned['REG'], valid)
        for k, a in idx.items():
            save_tif(dirs[k] / f'{name}_{k}.tif', [a], [k], {'radiometric_level': level})
            save_index_png(a, f'{name} {k} ({level})', dirs[k] / f'{name}_{k}.png',
                           -0.2 if k == 'NDVI' else -0.1, 1.0 if k == 'NDVI' else 0.6)
            v = a[np.isfinite(a)]
            row.update({f'{k}_mean': float(v.mean()), f'{k}_median': float(np.median(v)),
                        f'{k}_p05': float(np.percentile(v, 5)), f'{k}_p95': float(np.percentile(v, 95))})
            row.update({f'{k}_{kk}': vv for kk, vv in class_breakdown(a, INDEX_CLASSES[k]).items()})
        if {'NDVI', 'NDRE'} <= idx.keys():
            veg = idx['NDVI'] > 0.4
            row['NDRE_gt_NDVI_on_veg_%'] = float(100 * np.mean(idx['NDRE'][veg] > idx['NDVI'][veg])) if veg.any() else np.nan
            if row['NDRE_gt_NDVI_on_veg_%'] > 5:
                print(f'  WARNING {name}: NDRE > NDVI on {row["NDRE_gt_NDVI_on_veg_%"]:.1f}% of vegetation '
                      'pixels; radiometry is probably wrong')
        index_rows.append(row)
        c['indices'] = idx

    pd.DataFrame(align_rows).to_csv(out_root / 'alignment_summary.csv', index=False)
    summary = pd.DataFrame(index_rows)
    summary.to_csv(out_root / 'indices_summary.csv', index=False)
    with open(out_root / 'PROCESSING_REPORT.txt', 'w') as f:
        f.write('PARROT SEQUOIA: CALIBRATED + ALIGNED NDVI / NDRE\n' + '=' * 70 + '\n')
        f.write(f'Input folder        : {folder}\nReference band      : {SEQ_REFERENCE_BAND}\n')
        f.write('Radiometry          : I = f^2 (DN - B) / (A t ISO + C) / V(x,y)   [Parrot SensorModel, vignetting]\n')
        f.write(f'Illumination        : {"Sunshine-normalised to flight median" if NORMALISE_ILLUMINATION else "none"}\n')
        f.write(f'Panel calibration   : {"yes" if K_panel else "NO -> indices are not surface reflectance"}\n')
        f.write('Geometry            : ECC homography per band on gradient images (raw frames)\n\n')
        f.write(pd.DataFrame(align_rows).drop(columns='homography').to_string(index=False) + '\n\n')
        f.write(summary.to_string(index=False) + '\n')
    print(summary.round(3).to_string(index=False))
    print(f'\nOutputs: {out_root}')
    return cap, summary

In [ ]:
if __name__ == '__main__' or IN_COLAB:
    if os.environ.get('SEQ_SKIP_RUN') != '1':
        seq_captures, seq_summary = run_sequoia_pipeline()

## Package results

In [ ]:
import shutil
bundle = Path('/content/Combined_Results') if IN_COLAB else Path('Combined_Results')
bundle.mkdir(exist_ok=True)
for src, name in ((RESULTS, 'Segmentation'), (SEQ_OUT_ROOT, 'Sequoia_Indices')):
    if Path(src).exists():
        shutil.copytree(src, bundle / name, dirs_exist_ok=True)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('archive:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)